# NB02 — Potsdam Quantitative Eval

**Runs on Kaggle T4 GPU.**

**Datasets to attach:**
- `dummyirl/sam3-weights` — SAM3 checkpoint
- `dummyirl/6isprs` — Potsdam ISPRS GeoTIFF + labels

**What this does:**
1. Installs env (conda + mmcv + mmseg, ~15 min)
2. Clones `HarishDeepak/SegEarth-OV-3` (our improved fork)
3. Stages Potsdam val tile `6_15` at the path `cfg_potsdam.py` expects
4. Runs `eval.py configs/cfg_potsdam.py` → prints mIoU / mAcc
5. Displays per-class IoU table

> Val tile is always `6_15` (tile-level split, no random split — patch overlap would leak).

## 1 — Environment setup (~15 min, run once per session)

In [55]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -p /kaggle/working/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/kaggle/working/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

ERROR: File or directory already exists: '/kaggle/working/miniconda'
If you want to update an existing installation, use the -u option.
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
conda 26.3.2


In [56]:
!/kaggle/working/miniconda/bin/conda create -n segearth python=3.10 -y

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - defaults
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 26.3.2
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /kaggle/working/miniconda/envs/segearth

  added / updated specs:
    - python=3.10


The following NEW packages will be INSTALLED:

  _libgcc_mutex      pkgs/main/linux-64::_libgcc_mutex-0.1-main 
  _openmp_mutex      pkgs/main/linux-64::_openmp_mutex-5.1-52_gnu 
  bzip2              pkgs/main/linux-64::bzip2-1.0.8-h5eee18b_6 
  ca-certificates    pkgs/main/linux-64::ca-certificates-2026.5.14-h06a4308_0 
  ld_impl_linux-64   pkgs/main/linux-64::ld_impl_linux-64-2.44-h9e0c5a2_3 
  libexpat           pkgs/main/linux-64::libexpat-2.8.1-h7354ed3_1 
  libffi             pkgs/main/linux-64::libffi-3.4.8-h06d3fd0_3 
  libgcc             pkgs/m

In [57]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [58]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [59]:
%%bash
# mmseg 1.2.2 declares mmcv<2.2.0 as max; patch so version check passes
source /kaggle/working/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/kaggle/working/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX → 2.3.0")
EOF
pip install numpy==1.26.4 -q

Patched MMCV_MAX → 2.3.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [60]:
%%bash
source /kaggle/working/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmcv.ops import point_sample; print("MMCV OPS OK")
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

MMCV: 2.2.0
MMCV OPS OK
MMSEG OK
CUDA: True


/kaggle/working/miniconda/envs/segearth/lib/python3.10/site-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


## 2 — Clone our fork

In [61]:
import subprocess, os
from pathlib import Path

REPO = Path("/kaggle/working/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated → {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/rg-segearth-ov3.git", str(REPO)],
        check=True)
    print(f"Cloned → {REPO}")

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

Already up to date.
Updated → /kaggle/working/SegEarth-OV-3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openxlab 0.1.3 requires rich~=13.4.2, but you have rich 15.0.0 which is incompatible.


## 3 — Stage Potsdam val tile

`cfg_potsdam.py` expects:
```
/kaggle/working/PotsdamEval/
  img_dir/val/         ← *_RGB.tif  (val tile 6_15)
  ann_dir_indexed/val/ ← *_label_noBoundary.tif
```

In [62]:
from pathlib import Path

POTSDAM_INPUT = Path("/kaggle/input/datasets/dummyirl/6isprs")
EVAL_ROOT     = Path("/kaggle/working/PotsdamEval")
VAL_TILE      = "6_15"

img_dst   = EVAL_ROOT / "img_dir/val"
label_dst = EVAL_ROOT / "ann_dir_indexed/val"
img_dst.mkdir(parents=True, exist_ok=True)
label_dst.mkdir(parents=True, exist_ok=True)

img_files   = sorted(POTSDAM_INPUT.rglob(f"*{VAL_TILE}*_RGB.tif"))
label_files = sorted(POTSDAM_INPUT.rglob(f"*{VAL_TILE}*_label_noBoundary.tif"))

if not img_files:
    print("No val tile images found. Contents of Potsdam dataset:")
    for p in sorted(POTSDAM_INPUT.rglob("*.tif"))[:20]: print(" ", p.name)
else:
    for p in img_files:
        dst = img_dst / p.name
        if not dst.exists(): dst.symlink_to(p)
    for p in label_files:
        dst = label_dst / p.name
        if not dst.exists(): dst.symlink_to(p)
    print(f"Images:  {len(list(img_dst.iterdir()))}")
    print(f"Labels:  {len(list(label_dst.iterdir()))}")
    print("Staging done.")

Images:  1
Labels:  1
Staging done.


## 3b — Convert RGB labels → indexed (0–5)

`_label_noBoundary.tif` uses RGB colors, not class indices. This converts them so `eval.py` can read them.

In [63]:
import numpy as np
from PIL import Image
from pathlib import Path

RGB_TO_IDX = {
    (255, 255, 255): 0,  # impervious surface
    (  0,   0, 255): 1,  # building
    (  0, 255, 255): 2,  # low vegetation
    (  0, 255,   0): 3,  # tree
    (255, 255,   0): 4,  # car
    (255,   0,   0): 5,  # clutter
}

label_dst = Path("/kaggle/working/PotsdamEval/ann_dir_indexed/val")

for lbl_path in sorted(label_dst.iterdir()):
    rgb = np.array(Image.open(lbl_path).convert("RGB"))
    idx = np.zeros(rgb.shape[:2], dtype=np.uint8)
    for color, cls in RGB_TO_IDX.items():
        mask = (rgb[:,:,0] == color[0]) & (rgb[:,:,1] == color[1]) & (rgb[:,:,2] == color[2])
        idx[mask] = cls
    lbl_path.unlink()
    Image.fromarray(idx).save(lbl_path)

print(f"Converted {len(list(label_dst.iterdir()))} label(s) to indexed.")
print(f"Unique class indices: {np.unique(idx)}")

Converted 1 label(s) to indexed.
Unique class indices: [0]


## 4 — Run quantitative eval

In [2]:
%%bash
source /kaggle/working/miniconda/bin/activate segearth
cd /kaggle/working/SegEarth-OV-3
MPLBACKEND=Agg python eval.py configs/cfg_potsdam.py 2>&1 | tee /kaggle/working/potsdam_eval_log.txt

python3: can't open file '/kaggle/working/eval.py': [Errno 2] No such file or directory


bash: line 1: /kaggle/working/miniconda/bin/activate: No such file or directory
bash: line 2: cd: /kaggle/working/SegEarth-OV-3: No such file or directory


## 5 — Display results

In [65]:
import re
from pathlib import Path

log = Path("/kaggle/working/potsdam_eval_log.txt").read_text(errors="replace")

# Print last 50 lines (where MMSeg prints the metrics table)
lines = log.strip().splitlines()
print("\n".join(lines[-50:]))

        'mIoU',
        'mFscore',
    ], type='IoUMetric')
test_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations'),
    dict(type='PackSegInputs'),
]
vis_backends = [
    dict(type='LocalVisBackend'),
]
visualizer = dict(
    alpha=0.7,
    name='visualizer',
    type='SegLocalVisualizer',
    vis_backends=[
        dict(type='LocalVisBackend'),
    ])
work_dir = './work_dirs/cfg_potsdam'

Traceback (most recent call last):
  File "/kaggle/working/SegEarth-OV-3/eval.py", line 347, in <module>
    main()
  File "/kaggle/working/SegEarth-OV-3/eval.py", line 290, in main
    .from_cfg(cfg)
  File "/kaggle/working/miniconda/envs/segearth/lib/python3.10/site-packages/mmengine/runner/runner.py", line 462, in from_cfg
    runner = cls(
  File "/kaggle/working/miniconda/envs/segearth/lib/python3.10/site-packages/mmengine/runner/runner.py", line 416, in __init__
    self.visualizer = self.build_visualizer(visualizer)
  File "/kaggle/working/miniconda/envs/segeart